In [7]:
import pandas as pd
import re
import requests
from bs4 import BeautifulSoup
import time, json


In [8]:
base_url = "https://www.pokernews.com/tours/wsop/2026-wsop/2026-wsop-main-event/"
day9  = scrape_day(15923, base_url + "day9/")
day10 = scrape_day(15924, base_url + "day10/")
day11 = scrape_day(15925, base_url + "day11/")

In [13]:
def extract_hands(html):
    soup = BeautifulSoup(html, "html.parser")
    out = []
    for post in soup.select("main.event__post__body") or [soup]:
        current = None
        for p in post.find_all("p"):
            if p.find("sub"):
                continue
            text = p.get_text(" ", strip=True)
            m = re.match(r"Hand #(\d+):\s*(.*)", text)
            if m:
                current = {"hand": int(m.group(1)), "description": m.group(2)}
                out.append(current)
            elif current is not None:
                current["description"] += " " + text
    return out

In [12]:
def scrape_day(day_id, page_url):
    headers = {"User-Agent": "Mozilla/5.0"}
    rows = {}
    anchor = False
    seen = set()

    for _ in range(50):
        u = ("https://www.pokernews.com/reporting-uroborus.json.php"
             f"?v2&day_id={day_id}&event_id=5587&filter=%5B%5D"
             f"&ts_anchor={anchor}&offset=10&page=0")
        r = requests.get(u, headers=headers)
        r.raise_for_status()
        j = r.json()

        payload = j["data"]
        html = payload if isinstance(payload, str) else json.dumps(payload)
        html = html.encode().decode("unicode_escape") if "\\u" in html else html
        for h in extract_hands(html):
            rows[h["hand"]] = h

        nxt = j["ts_anchor"]
        if not nxt or nxt in seen:
            break
        seen.add(nxt)
        anchor = nxt
        time.sleep(1)

    page = requests.get(page_url, headers=headers)
    page.raise_for_status()
    for h in extract_hands(page.text):
        rows[h["hand"]] = h

    return pd.DataFrame(rows.values()).sort_values("hand").reset_index(drop=True)

In [11]:
all_hands = (pd.concat([day9, day10, day11])
               .drop_duplicates(subset="hand")
               .sort_values("hand")
               .reset_index(drop=True))
all_hands.to_csv("all_hands.csv", index=False)
